In [ ]:
import torch
import torch_geometric
import pandas as pd
import sklearn


print("Libraries loaded successfully.")

In [ ]:
# data loading and preprocessing

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

print("Loading data...")
train_transaction = pd.read_csv('data/train_transaction.csv')
train_identity = pd.read_csv('data/train_identity.csv')

df = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

df_sample = df.sample(n=50000, random_state=42)

features = [
    'TransactionAmt', 'ProductCD', 'card1', 'card4', 'addr1',
    'P_emaildomain', 'DeviceType', 'isFraud'
]
df_subset = df_sample[features].copy()

df_subset.dropna(inplace=True)

card_encoder = LabelEncoder()
addr_encoder = LabelEncoder()
df_subset['card_id'] = card_encoder.fit_transform(df_subset['card1'])
df_subset['addr_id'] = addr_encoder.fit_transform(df_subset['addr1'])

df_features = pd.get_dummies(df_subset, columns=['ProductCD', 'DeviceType'])
cat_cols = [c for c in df_features.columns if c.startswith('ProductCD_') or c.startswith('DeviceType_')]

aggs = {'TransactionAmt': ['mean', 'std', 'max', 'count']}
for c in cat_cols:
    aggs[c] = ['sum']

card_features_df = df_features.groupby('card_id').agg(aggs).fillna(0)
addr_features_df = df_features.groupby('addr_id').agg(aggs).fillna(0)

scaler = StandardScaler()
card_features_scaled = scaler.fit_transform(card_features_df)
addr_features_scaled = scaler.transform(addr_features_df)

print("\nData preprocessing, Feature Engineering & Scaling complete.")


In [ ]:
# heterogeneous graph construction

import torch
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

data = HeteroData()

num_card_nodes = card_features_scaled.shape[0]
num_addr_nodes = addr_features_scaled.shape[0]

data['card'].x = torch.tensor(card_features_scaled, dtype=torch.float32)
data['addr'].x = torch.tensor(addr_features_scaled, dtype=torch.float32)

source_nodes = torch.tensor(df_subset['card_id'].values, dtype=torch.long)
destination_nodes = torch.tensor(df_subset['addr_id'].values, dtype=torch.long)
edge_index = torch.stack([source_nodes, destination_nodes], dim=0)
data['card', 'uses', 'addr'].edge_index = edge_index

data = T.ToUndirected()(data)

card_labels = df_subset.groupby('card_id')['isFraud'].max().sort_index()
data['card'].y = torch.tensor(card_labels.values, dtype=torch.long)

# 80/20 train/test split
indices = torch.randperm(num_card_nodes, generator=torch.Generator().manual_seed(42))
split_idx = int(num_card_nodes * 0.8)
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

data['card'].train_mask = torch.zeros(num_card_nodes, dtype=torch.bool)
data['card'].train_mask[train_idx] = True
data['card'].test_mask = torch.zeros(num_card_nodes, dtype=torch.bool)
data['card'].test_mask[test_idx] = True

print("Heterogeneous graph constructed with node masks and engineered features.")
print(data)


In [ ]:
# gnn model

import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, to_hetero

class GNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        # we are using -1 for in_channels because it then allows PyG to infer the input size
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model_base = GNN(hidden_channels=64, out_channels=2)

# automatically creates different message passing functions for each edge type
model = to_hetero(model_base, data.metadata(), aggr='sum')

print("Heterogeneous GNN model defined.")
print("\nModel Architecture:")
print(model)

In [ ]:
# training
if torch.backends.mps.is_available(): device = torch.device('mps')
elif torch.cuda.is_available(): device = torch.device('cuda')
else: device = torch.device('cpu')

data, model = data.to(device), model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

train_y = data['card'].y[data['card'].train_mask]
num_fraud = train_y.sum().item()
num_legit = train_y.shape[0] - num_fraud
weight = torch.tensor([1.0, num_legit / max(num_fraud, 1)], dtype=torch.float32).to(device)

print("\nStarting model training...")
for epoch in range(1, 5001):
    model.train()
    optimizer.zero_grad()
    out = model(data.x_dict, data.edge_index_dict)
    
    out_train = out['card'][data['card'].train_mask]
    y_train = data['card'].y[data['card'].train_mask]
    
    loss = torch.nn.functional.cross_entropy(out_train, y_train, weight=weight)
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:02d}, Loss: {loss.item():.4f}')
print("\nModel training complete.")


In [ ]:
# evaluation

from sklearn.metrics import roc_auc_score

@torch.no_grad()
def evaluate_model():
    model.eval()
    out = model(data.x_dict, data.edge_index_dict)['card']
    
    # filter
    out_test = out[data['card'].test_mask]
    y_test = data['card'].y[data['card'].test_mask]
    
    pred_prob = out_test.softmax(dim=-1)[:, 1]
    
    auc = roc_auc_score(y_test.cpu().numpy(), pred_prob.cpu().numpy())
    return auc

final_auc = evaluate_model()
print(f'\nROC AUC Score: {final_auc:.4f}')
